In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.4 MB/s eta 0:00:00


In [3]:
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 5.7 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import numpy as np
import pandas as pd
import faiss
import imagehash
from PIL import Image

In [6]:
EMB_DIR = "/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings"

embeddings = np.load(f"{EMB_DIR}/clip_embeddings.npy").astype("float32")
meta = pd.read_csv(f"{EMB_DIR}/clip_embeddings_meta.csv")

index = faiss.read_index("/content/drive/MyDrive/copydays-ndid/Nagarjuna/ndid_faiss.index")
K = 10
DUPLICATE_THRESHOLD = 0.82
PHASH_THRESHOLD = 5

In [ ]:
def classify_duplicate(query_idx):
    q = embeddings[query_idx].reshape(1, -1)
    sims, idxs = index.search(q, K)

    # exclude self (rank 0)
    sims = sims[0][1:]
    idxs = idxs[0][1:]

    best_sim = sims[0]

    if best_sim >= DUPLICATE_THRESHOLD:
        decision = "DUPLICATE"
    else:
        decision = "NOT DUPLICATE"

    return decision, float(best_sim), idxs.tolist()

In [ ]:
query_idx = 123

decision, sim, neighbors = classify_duplicate(query_idx)

print("Decision:", decision)
print("Best similarity:", sim)

print("\nTop neighbors:")
for i in neighbors[:5]:
    print(meta.iloc[i]["image_path"], "| group:", meta.iloc[i]["group_id"])

Decision: DUPLICATE
Best similarity: 0.9722933769226074

Top neighbors:
/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/augmented/IMAGE_00024/original.jpg | group: IMAGE_00024
/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/augmented/IMAGE_00024/brightened.jpg | group: IMAGE_00024
/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/augmented/IMAGE_00024/blurred.jpg | group: IMAGE_00024
/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/augmented/IMAGE_00024/cropped.jpg | group: IMAGE_00024
/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/augmented/IMAGE_05726/cropped.jpg | group: IMAGE_05726
